### video 2 - ai parse document

In [0]:
-- setup temp table to stage the data before transformation
create or replace temp view raw_unstructure_doc as 
select
  path,
  content
from
  read_files('/Volumes/idp/default/youtube_lesson')

In [0]:
select 
  path, length(content)
from
  raw_unstructure_doc

In [0]:
create or replace temp view parsed_structured_docs as

select
  path,
  ai_parse_document(content) as parsed_content
from
  raw_unstructure_doc

In [0]:
select path, parsed_content from parsed_structured_docs

In [0]:
create or replace temp view structured_tables as 
select 
  path,
  try_cast(e:content as string) as table_html
from
  parsed_structured_docs
lateral view 
  explode(try_cast(parsed_content:document:elements as array<variant>)) t as e
where
  try_cast(e:type as string) = 'table'

In [0]:
select
  *
from
  structured_tables

### video 3 - ai extract


In [0]:
select
  ai_extract(table_html, array('CPT Code')).`CPT Code` as CPT_Code,
  ai_extract(table_html, array('ICD Code')).`ICD Code` as ICD_Code,
  ai_extract(table_html, array('Description')).`Description` as Description,
  ai_extract(table_html, array('Billed Amount')).`Billed Amount` as Billed_Amount,
  ai_extract(table_html, array('Paid Amount')).`Paid Amount` as Paid_Amount
from
  structured_tables

In [0]:
with rows as (
  select
    *,
    explode(regexp_extract_all(table_html, '<tr>(.*?)</tr>', 1)) as tr
  from
    structured_tables
), data_rows as (
  select
    path,
    tr
  from
    rows
  where
    tr like '%<td%'
)
select
  ai_extract(tr, array('CPT Code')).`CPT Code` as CPT_Code,
  ai_extract(tr, array('ICD Code')).`ICD Code` as ICD_Code,
  ai_extract(tr, array('Description')).`Description` as Description,
  ai_extract(tr, array('Billed Amount')).`Billed Amount` as Billed_Amount,
  ai_extract(tr, array('Paid Amount')).`Paid Amount` as Paid_Amount
from
  data_rows